In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

@njit
def evaluate_faces(X, faces, active_faces, theta):
    ruptured = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])
            if frustration > theta:
                active_faces[f] = False
                ruptured += 1
    return ruptured

# ==========================================
# 2. 仮説3：高次元介入（バイパス手術）のシミュレーション
# ==========================================
def run_hypothesis3_simulation(N, m, params):
    dt = params['dt']
    T1, T2, T3, T4 = params['T1'], params['T2'], params['T3'], params['T4']

    row, col, faces = build_simplicial_complex(N, m)
    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    del row, col, data
    gc.collect()

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []

    ruptured_faces_count = 0
    injected_faces_count = 0 # Phase 3でアルゴリズムが戦略的に注入する2-単体の数(バイパス)

    for step in tqdm(range(T4), desc=f"Simulating N={N}", leave=False):
        # フェーズ管理
        if step == T1:
            b[hub_node] = 0.9 # Phase 1a: フェイク投下（Pathogenesis）
        elif step == T2:
            b[hub_node] = 0.0 # Phase 1b: フェイク除去（Hysteresis）

        elif step == T3:
            # Phase 3: 高次元アルゴリズム介入（局所的バイパス手術）
            # プラットフォームが、壊死領域の周辺でフラストレーションが安全な閾値以下の
            # トライアド(3者間)を特定し、破裂した面数の95%に相当するM'個の2-単体を戦略的に注入する。
            M_prime = int(ruptured_faces_count * 0.95)
            injected_faces_count += M_prime

        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        # 既存面の破裂を継続評価
        ruptured_faces_count += evaluate_faces(X, faces, active_faces, params['theta'])

        # Euler-Poincaré: Δβ1 = - ΔF
        # 破裂した面（減少）と、アルゴリズムが注入した面（増加）の差し引きで穴の数を計算
        current_delta_betti_1 = max(0, ruptured_faces_count - injected_faces_count)
        history_betti_1.append(current_delta_betti_1 / max(1, num_faces))

    return history_betti_1

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000, 10000000]

    params = {
        'dt': 0.1,
        'T1': 100,     # Phase 0: Burn-in終了
        'T2': 200,     # Phase 1a: フェイク投下終了
        'T3': 300,     # Phase 1b: ヒステリシス確認終了 -> Phase 3 介入開始
        'T4': 400,     # Phase 3: 高次元介入の効果確認終了
        'epsilon': 0.05,
        'sigma': 0.02,
        'theta': 0.25  # 面の破裂閾値
    }

    output_dir = "Simplicial_TMFG_Hypothesis3"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] 仮説3 (高次元介入・バイパス手術) シミュレーション開始...")
            betti_1_history = run_hypothesis3_simulation(N, m=3, params=params)

            # CSV出力
            df = pd.DataFrame({
                'TimeStep': np.arange(params['T4']),
                'Delta_Betti_1_Ratio': betti_1_history
            })
            df.to_csv(os.path.join(output_dir, f'h3_result_N{N}.csv'), index=False)

            # グラフの出力
            plt.figure(figsize=(12, 6))
            plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'g-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')

            # 背景色でフェーズを可視化
            plt.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in (Self-Organization)')
            plt.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Pathogenesis (Fake Exposed)')
            plt.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Hysteresis (Irreversible Damage)')
            plt.axvspan(params['T3'], params['T4'], color='green', alpha=0.2, label='Phase 3: High-Dimensional Intervention (Bypass Surgery)')

            plt.title(f'Success of High-Dimensional Algorithmic Intervention (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')
            plt.legend(loc='upper left')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'h3_plot_N{N}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            summary_data.append({
                'N': N,
                'Peak_Damage_Ratio': betti_1_history[params['T3'] - 1],
                'Final_Betti_1_Ratio': betti_1_history[-1]
            })

            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。ここまでのデータでZIP化へ移行します。")
            break

    # まとめグラフとZIP化
    if len(summary_data) > 0:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'h3_summary.csv'), index=False)

        # 介入前(Peak)と介入後(Final)の効果を比較するグラフ
        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Peak_Damage_Ratio'], 'r--o', linewidth=2, label='Pre-Intervention (Hysteresis Peak)')
        plt.plot(df_sum['N'], df_sum['Final_Betti_1_Ratio'], 'g-o', linewidth=2, label='Post-Intervention (Recovered)')
        plt.xscale('log')
        plt.title('Compensatory Recovery via High-Dimensional Intervention')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Proportion of Topological Holes')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'h3_summary_recovery.png'), dpi=300, bbox_inches='tight')
        plt.close()

        zip_filename = 'Simplicial_TMFG_Hypothesis3.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)

        print(f"\n🎉 仮説3の計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:169: SyntaxWarning: invalid escape sequence '\D'
<>:179: SyntaxWarning: invalid escape sequence '\D'
<>:169: SyntaxWarning: invalid escape sequence '\D'
<>:179: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_551/4005992604.py:169: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'g-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
/tmp/ipykernel_551/4005992604.py:179: SyntaxWarning: invalid escape sequence '\D'
  plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')



[1,000 Nodes] 仮説3 (高次元介入・バイパス手術) シミュレーション開始...



[10,000 Nodes] 仮説3 (高次元介入・バイパス手術) シミュレーション開始...



[100,000 Nodes] 仮説3 (高次元介入・バイパス手術) シミュレーション開始...



[1,000,000 Nodes] 仮説3 (高次元介入・バイパス手術) シミュレーション開始...



[10,000,000 Nodes] 仮説3 (高次元介入・バイパス手術) シミュレーション開始...



🎉 仮説3の計算完了！結果は 'Simplicial_TMFG_Hypothesis3.zip' に出力されました。


In [2]:
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

@njit
def evaluate_faces(X, faces, active_faces, theta):
    ruptured = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])
            if frustration > theta:
                active_faces[f] = False
                ruptured += 1
    return ruptured

# ==========================================
# 2. 仮説3 シミュレーション (自己組織化のKS検定付き)
# ==========================================
def run_hypothesis3_with_validation(N, m, params):
    dt = params['dt']
    T1, T2, T3, T4 = params['T1'], params['T2'], params['T3'], params['T4']

    row, col, faces = build_simplicial_complex(N, m)
    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    del row, col, data
    gc.collect()

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []
    history_ks_stat = []
    X_snapshots = {}

    ruptured_faces_count = 0
    injected_faces_count = 0
    X_prev = np.copy(X)

    for step in tqdm(range(T4), desc=f"Simulating N={N}", leave=False):
        if step == T1:
            b[hub_node] = 0.9
        elif step == T2:
            b[hub_node] = 0.0
        elif step == T3:
            # 高次元アルゴリズム介入: 破裂した面数の95%に相当する新しい2-単体(バイパス)を注入
            M_prime = int(ruptured_faces_count * 0.95)
            injected_faces_count += M_prime

        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        ruptured_faces_count += evaluate_faces(X, faces, active_faces, params['theta'])

        # Euler-Poincaré: Δβ1 = - ΔF (破裂した面数 - 注入された面数)
        current_delta_betti_1 = max(0, ruptured_faces_count - injected_faces_count)
        history_betti_1.append(current_delta_betti_1 / max(1, num_faces))

        # Burn-in期間のKS検定と分布スナップショット
        if step <= T1:
            if step % 10 == 0:
                ks_stat, _ = stats.ks_2samp(X[::100], X_prev[::100])
                history_ks_stat.append((step, ks_stat))
                X_prev = np.copy(X)

            if step in [0, int(T1/2), T1]:
                X_snapshots[step] = np.copy(X)

    return history_betti_1, history_ks_stat, X_snapshots

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000]

    params = {
        'dt': 0.1,
        'T1': 100,     # Phase 0: Burn-in終了
        'T2': 200,     # Phase 1a: フェイク投下終了
        'T3': 300,     # Phase 1b: ヒステリシス確認終了 -> Phase 3 介入開始
        'T4': 400,     # Phase 3: 高次元介入の効果確認終了
        'epsilon': 0.05,
        'sigma': 0.02,
        'theta': 0.25
    }

    output_dir = "Hypothesis3_Empirical_Validation"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] 仮説3 (高次元介入・バイパス手術) ＋ 自己組織化検証 開始...")
            betti_1_hist, ks_hist, snapshots = run_hypothesis3_with_validation(N, 3, params)

            # --- グラフ1: KS統計量による自己組織化 ---
            steps_ks, ks_vals = zip(*ks_hist)
            plt.figure(figsize=(8, 5))
            plt.plot(steps_ks, ks_vals, 'g-o', linewidth=2)
            plt.title(f'Convergence to Empirical Stationary Distribution (N={N})')
            plt.xlabel('Burn-in Steps')
            plt.ylabel('KS Statistic (Distribution Drift)')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'H3_KS_Convergence_N{N}.png'), dpi=300)
            plt.close()

            # --- グラフ2: 状態分布の進化 ---
            plt.figure(figsize=(8, 5))
            for stp, x_data in snapshots.items():
                plt.hist(x_data, bins=50, alpha=0.5, density=True, label=f'Step {stp}')
            plt.title(f'Self-Organization of Conformity Distribution (N={N})')
            plt.xlabel('Conformity (X)')
            plt.ylabel('Density')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'H3_Distribution_Evolution_N{N}.png'), dpi=300)
            plt.close()

            # --- グラフ3: 仮説3 高次元介入による回復 ---
            df = pd.DataFrame({'TimeStep': np.arange(params['T4']), 'Delta_Betti_1_Ratio': betti_1_hist})
            df.to_csv(os.path.join(output_dir, f'h3_betti1_result_N{N}.csv'), index=False)

            plt.figure(figsize=(12, 6))
            plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'g-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
            plt.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in (Self-Organization)')
            plt.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Fake News Exposed')
            plt.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Hysteresis (Irreversible Damage)')
            plt.axvspan(params['T3'], params['T4'], color='green', alpha=0.2, label='Phase 3: High-Dimensional Intervention (Bypass)')

            plt.title(f'Success of High-Dimensional Algorithmic Intervention (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')
            plt.legend(loc='upper right')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'H3_Topological_Recovery_N{N}.png'), dpi=300)
            plt.close()

            summary_data.append({
                'N': N,
                'Peak_Damage_Ratio': betti_1_hist[params['T3'] - 1],
                'Final_Betti_1_Ratio': betti_1_hist[-1]
            })
            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # まとめZIP化
    if summary_data:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'h3_summary_all_N.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Peak_Damage_Ratio'], 'r--o', linewidth=2, label='Pre-Intervention (Hysteresis Peak)')
        plt.plot(df_sum['N'], df_sum['Final_Betti_1_Ratio'], 'g-o', linewidth=2, label='Post-Intervention (Recovered)')
        plt.xscale('log')
        plt.title('Compensatory Recovery via High-Dimensional Intervention')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Proportion of Topological Holes')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'H3_summary_recovery_plot.png'), dpi=300)
        plt.close()

        zip_filename = 'Hypothesis3_Empirical_Validation.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)
        print(f"\n🎉 仮説3の計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()


[1,000 Nodes] 仮説3 (高次元介入・バイパス手術) ＋ 自己組織化検証 開始...


<>:195: SyntaxWarning: invalid escape sequence '\D'
<>:203: SyntaxWarning: invalid escape sequence '\D'
<>:195: SyntaxWarning: invalid escape sequence '\D'
<>:203: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_551/3090976735.py:195: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'g-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
/tmp/ipykernel_551/3090976735.py:203: SyntaxWarning: invalid escape sequence '\D'
  plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')



[10,000 Nodes] 仮説3 (高次元介入・バイパス手術) ＋ 自己組織化検証 開始...



[100,000 Nodes] 仮説3 (高次元介入・バイパス手術) ＋ 自己組織化検証 開始...



[1,000,000 Nodes] 仮説3 (高次元介入・バイパス手術) ＋ 自己組織化検証 開始...



🎉 仮説3の計算完了！結果は 'Hypothesis3_Empirical_Validation.zip' に出力されました。
